## Loading data

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

DB_PATH = Path("route_all.db")   

if not DB_PATH.exists():
    raise FileNotFoundError(f"{DB_PATH.resolve()} not found. Fix DB_PATH or move the notebook next to the file.")

conn = sqlite3.connect(DB_PATH)
conn.execute("CREATE INDEX IF NOT EXISTS idx_route ON stoptimes(route)")
conn.commit()

In [18]:
ONLY_REALTIME = True                 # False = also keep rows where is_realtime = 0
RUSH_HOURS = (7, 8, 14, 15, 16)      # trip start hours counted as rush hour (weekdays only)
WEEKEND = ("La", "Su")
TRIP_KEYS = ["trip_id"]

# --- Query (no ORDER BY: sorting 9M rows in SQLite is slow; pandas does it below) ---
query = "SELECT route, trip_id, stop_position, stop_id, scheduled_departure_sec, delay_seconds FROM stoptimes"
if ONLY_REALTIME:
    query += " WHERE is_realtime = 1"

df = pd.read_sql_query(query, conn)

df = df.astype({
    "route": "category",
    "stop_id": "category",
    "trip_id": "category",
    "stop_position": "int16",
    "scheduled_departure_sec": "int32",
    "delay_seconds": "int32",   # NULLs in database appear as 0
})
df = df.sort_values(["route", "trip_id", "stop_position"], ignore_index=True)

# --- Drop trips whose delay is zero at every stop ---
has_delay = df["delay_seconds"].ne(0).groupby([df[k] for k in TRIP_KEYS], observed=True).transform("any")
df = df[has_delay].reset_index(drop=True)
for c in ("route", "trip_id"):
    df[c] = df[c].cat.remove_unused_categories()

print(f"{len(df):,} rows, {df.groupby(['route', 'trip_id'], observed=True).ngroups:,} trips "
      f"across {df['route'].nunique():,} route(s)")

# --- Parse trip_id once per unique trip, then make auxiliary columns ---
trip_cats = df["trip_id"].cat.categories.astype(str)
parts = pd.Series(trip_cats).str.split("_", expand=True)
codes = df["trip_id"].cat.codes.to_numpy()

def spread_category(col):
    c = parts[col].astype("category")
    return pd.Categorical.from_codes(c.cat.codes.to_numpy()[codes], c.cat.categories)

df["direction"] = spread_category(3)
df["weekday"] = spread_category(2)
df["hour"] = parts[4].str[:2].astype("int16").to_numpy()[codes] 

# --- Flags ---
df["is_weekend"] = df["weekday"].isin(WEEKEND).astype("int8")
df["is_rush_hour"] = ((df["is_weekend"] == 0) & df["hour"].isin(RUSH_HOURS)).astype("int8")

df.head()

3,546,121 rows, 115,752 trips across 362 route(s)


,route,trip_id,stop_position,stop_id,scheduled_departure_sec,delay_seconds,direction,weekday,hour,is_weekend,is_rush_hour
0,102N,HSL:2102N_20261009_La_1_2359,0,HSL:1040273,86340,12,1,La,23,1,0
1,102N,HSL:2102N_20261009_La_1_2359,1,HSL:1040233,86400,69,1,La,23,1,0
2,102N,HSL:2102N_20261009_La_1_2359,2,HSL:1201129,86580,0,1,La,23,1,0
3,102N,HSL:2102N_20261009_La_1_2359,3,HSL:1201131,86640,44,1,La,23,1,0
4,102N,HSL:2102N_20261009_La_1_2359,4,HSL:1310101,86760,22,1,La,23,1,0


## Linear models

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = "delay_seconds"
num_cols = ["stop_position", "is_rush_hour", "is_weekend"]
cat_cols = ["direction"]

RANDOM_SEED = 59
TEST_FRAC = 0.2

# --- Random split by trip_id ---
rng = np.random.default_rng(RANDOM_SEED)
n_trips = len(df["trip_id"].cat.categories)
test_trips = rng.choice(n_trips, size=int(n_trips * TEST_FRAC), replace=False)  # category codes

is_test = np.isin(df["trip_id"].cat.codes.to_numpy(), test_trips)
train, test = df[~is_test], df[is_test]
print(f"train {len(train):,} rows / {n_trips - len(test_trips):,} trips, "
      f"test {len(test):,} rows / {len(test_trips):,} trips")

def make_model(with_route: bool):
    cats = cat_cols + (["route"] if with_route else [])
    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cats),
    ])
    return Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0))])

def evaluate(model, name):
    cols = num_cols + cat_cols + ["route"]
    X_tr, X_te = train[cols].astype({c: str for c in cat_cols + ["route"]}), test[cols].astype({c: str for c in cat_cols + ["route"]})
    model.fit(X_tr, train[target])
    pred = model.predict(X_te)
    mae = mean_absolute_error(test[target], pred)
    r2 = r2_score(test[target], pred)
    print(f"{name:<22} MAE {mae:.3f}   R2 {r2:.3f}")
    return model

m_global = evaluate(make_model(with_route=False), "No route effect")
m_route  = evaluate(make_model(with_route=True),  "Route fixed effects")

# --- Median train (baseline) ------
name = "Median (baseline)"
pred = np.full(len(test), train[target].median())

mae = mean_absolute_error(test[target], pred)
r2 = r2_score(test[target], pred)

print(f"{name:<22} MAE {mae:.3f}   R2 {r2:.3f}")


train 2,836,821 rows / 92,602 trips, test 709,300 rows / 23,150 trips
No route effect        MAE 87.528   R2 0.014


c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Route fixed effects    MAE 85.035   R2 0.081
Median (baseline)      MAE 86.233   R2 -0.022


## Lightgbm

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

# ---------------- Settings ----------------
SEED = 59
TEST_FRAC, VAL_FRAC = 0.2, 0.1
target = "delay_seconds"

features = ["stop_position", "hour", "route", "direction", "weekday"]
data=df.copy()

# ---------------- Random split by trip_id ----------------
rng = np.random.default_rng(SEED)
codes = data["trip_id"].cat.codes.to_numpy()
perm = rng.permutation(np.unique(codes))
n_test = int(len(perm) * TEST_FRAC)
test_trips, rest = perm[:n_test], perm[n_test:]
n_val = int(len(rest) * VAL_FRAC)
val_trips, train_trips = rest[:n_val], rest[n_val:]

train = data[np.isin(codes, train_trips)]
val   = data[np.isin(codes, val_trips)]
test  = data[np.isin(codes, test_trips)]
print(f"train {len(train):,} rows | val {len(val):,} | test {len(test):,} rows")

# ---------------- Model ----------------
model = lgb.LGBMRegressor(
    objective="l1",
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=200,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    cat_smooth=20,
    random_state=SEED, n_jobs=-1, verbose=-1,
)
model.fit(
    train[features], train[target],
    eval_set=[(val[features], val[target])], eval_metric="l1",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)
print(f"best iteration: {model.best_iteration_}")

# ---------------- Evaluation ----------------
y = test[target].to_numpy()
preds = {
    "Baseline (train median)": np.full(len(test), train[target].median()),
    "LightGBM (l1)": model.predict(test[features]),
}


def metrics(p):
    e = y - p
    return pd.Series({
        "MAE": np.abs(e).mean(),
        "MedAE": np.median(np.abs(e)),
        "RMSE": np.sqrt((e ** 2).mean()),
        "R2 (test)": 1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum(),
        "Bias (pred - actual)": (p - y).mean(),
    })

P = pd.DataFrame(preds)
table = P.apply(metrics).T.sort_values("MAE")
table["MAE gain vs median baseline"] = 1 - table["MAE"] / table.loc["Baseline (train median)", "MAE"]
print("\n", table.round(3).to_string())

train 2,553,534 rows | val 284,852 | test 707,735 rows


c:\Users\Lenovo\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 77.5431
[400]	valid_0's l1: 77.1275
[600]	valid_0's l1: 76.7942
[800]	valid_0's l1: 76.4708
[1000]	valid_0's l1: 76.1289
[1200]	valid_0's l1: 75.8779
[1400]	valid_0's l1: 75.6404
[1600]	valid_0's l1: 75.3983
[1800]	valid_0's l1: 75.2553
[2000]	valid_0's l1: 75.1218
[2200]	valid_0's l1: 75.0122
[2400]	valid_0's l1: 74.9199
[2600]	valid_0's l1: 74.8385
[2800]	valid_0's l1: 74.7838
[3000]	valid_0's l1: 74.7536
Did not meet early stopping. Best iteration is:
[2924]	valid_0's l1: 74.7494
best iteration: 2924

                             MAE  MedAE     RMSE  R2 (test)  Bias (pred - actual)  MAE gain vs median baseline
LightGBM (l1)            76.247  49.52  140.348      0.138               -15.689                        0.124
Baseline (train median)  87.068  55.00  152.918     -0.023               -22.897                        0.000


In [23]:
import joblib

artifact = {
    "model": model,                                   
    "features": features,                             
    "target": target,
    "categories": {c: train[c].cat.categories.tolist()
                   for c in ["route", "direction", "weekday"]},  
    "best_iteration": model.best_iteration_,
}
joblib.dump(artifact, "lightgbm_bus_delay.joblib", compress=3)
print("saved lightgbm_bus_delay.joblib")

saved lightgbm_bus_delay.joblib
